# Water Main Break EDA

This notebook is a quick analysis of the water main break data with the end result being a Plotly scatter mapbox of all the breaks, with their colour being the year that they broke, and their size being the number of breaks they've experienced.

In previous notebooks, I cleaned up the data to get it ready for the ML models, but for this notebook I've imported the original dataset so that we have the necessary data to create the visuals we're looking for.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
break_data = pd.read_csv('../data/Nov_10_22_Water_Main_Breaks.csv')
break_data.sample(5)

## Data Cleaning

Here I'll take out well over half of the original features so we have something nicer to look at when dispalying the dataframe for inspection, but first we'll convert our X and Y variables to latitude and longitude and then convert all of the columns to lowercase.

In [ ]:
break_data.rename(columns={'X': 'lon', 'Y': 'lat'}, inplace=True)

In [ ]:
# convert column names to lower case
break_data.columns = map(str.lower, break_data.columns)  # convert column names to lower case
break_data.columns

In [ ]:
break_data = break_data[['lon', 'lat', 'objectid', 'incident_date', 'break_type', 'hour_impacted', 'units_impacted', 'status',
                         'status_date', 'break_nature', 'break_apparent_cause', 'break_categorization', 'roadsegmentid', 
                         'street', 'assetid', 'asset_size', 'asset_year_installed', 'asset_material']]

In [ ]:
break_data.head()

In [ ]:
break_data.columns

There, much better!

Now that we have a prettier looking dataset, let's convert the date column to the proper format using pandas `to_datetime()` method.

In [ ]:
break_data['incident_date'] = pd.to_datetime(break_data['incident_date'])  # convert incident date to date

In [ ]:
print(break_data['incident_date'].info())
break_data.head()

I feel like I can take out the year and week and make them their own separate columns. This sort of future proofs the date data if we come back to it and analyze breaks by time of year or something like that.

In [ ]:
break_data['week'] = break_data['incident_date'].dt.week  # add week column
break_data['year'] = break_data['incident_date'].dt.year  # add year column
break_data.head()

## Number of Breaks Per Year

It would be nice if we could see a plot of the number of breaks that have occurred each year. In order to do that we first need to know the number of breaks that actually do occur each year. Therefore we need to create a `breaks` or `num_breaks` column and calculate the number of times each pipe has broken.

To create our new column, let's use the pandas `groupby()` function and group by each pipe's asset ID, and basically we want to sum up each time we see the same ID to get a count for the number of times each pipe has broken in it's lifetime.

Logic:
- group by asset ID
    - on asset ID
- count each time we see the same ID

In [ ]:
# create new column for number of breaks for each assetid
break_data['breaks'] = break_data.groupby('assetid')['assetid'].transform('count')  # add breaks column

In [ ]:
break_data.sample(5)

Let's display the number of breaks per year now in a nice looking bar plot

In [ ]:
# src: https://www.folkstalk.com/tech/how-to-display-values-on-top-of-bar-in-barplot-seaborn-with-code-examples/
plt.figure(figsize=(12, 6))
ax = sns.countplot(x='year', data=break_data, palette='Blues_d')
ax.bar_label(ax.containers[0]) # this displays the numbers above the bars in the plot
plt.title('Number of Water Main Breaks per Year')
plt.xlabel('Year')
plt.xticks(rotation=45)
plt.ylabel('Number of Breaks')
plt.show();

## Breaks by Location

What I'm looking for in this project is to be able to display a heatmap of the pipes that are the most at high risk of breaking based on their previous break frequency. 

A good starting point would be to see where the most breaks have occurred already, so I need to display the water main breaks by location and be able to visualize where in the city are the worst areas.

Let's first see in a scatter plot the latitude and longitudes of the breaks.

In [ ]:
plt.figure(figsize=(12, 10))
sns.scatterplot(x="lon", y="lat", data=break_data)
plt.title('Breaks by Location')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show();

Now let's overlay these points onto an actual map of Kitchener-Waterloo.

I'm familiar with the gmaps library as I've worked on another project using the API, so I'll be using it again here to show the heatmap.

In [ ]:
import config
import gmaps
import gmaps.datasets
gmaps.configure(api_key=config.gmaps_api_key)

To display the heatmap, `gmaps` needs the location data either in a dataset with just those two features or have the latitude and longitude as a paired list. What I do is put the values of latitude and longitude in their own variable and then create a paired list with the `list(zip())` method and store that as its own variable as well.

We can take the break locations paired list and pass that into `gmaps.heatmap_layer()`, and then add that layer to the gmaps figure instantiated before, and that's how you get our final output!

In [ ]:

lat = break_data['lat'].values
lon = break_data['lon'].values
break_locations = list(zip(lat, lon))
fig = gmaps.figure()
break_map = gmaps.heatmap_layer(break_locations)
fig.add_layer(break_map)
fig

Sometimes, trying to display with the gmaps library doesn't work. So I have downloaded the image and displayed it using IPython below.

In [ ]:
from IPython.display import Image
Image(filename='../data/break_map.png')

Another great API for displaying geographic data is Mapbox. It works in conjunction with plotly so it's quite easy to use, as long as you have an access token.

I'll show how we can integrate this powerful API with our data below!

In [ ]:
import plotly.express as px
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)

First I have to pass in my unique access token so when we call to the API, the image can get renered.

I then set the scatter map from plotly to a variable and define all of its attributes appropriately. It's easier to set the map to a variable so I can make different calls to it to update it, rather than have one long chain of methods.

The type of map used is defined in the `update_layout()` method and then we go ahead and render the image.

In [ ]:
token = config.token
px.set_mapbox_access_token(token)
fig = px.scatter_mapbox(data_frame=break_data, lat='lat', lon='lon',
                        color='year', size='breaks', hover_name='street',
                        color_continuous_scale=px.colors.cyclical.IceFire)
fig.update_layout(mapbox_style='carto-positron', mapbox_accesstoken=token)
fig.show(renderer='notebook_connected')

When you hover over each point you'll see the street name as the header, the number of breaks the pipe has experienced in its lifetime, the location in latitude/longitude and the year of it's last break. 

The size of each point corresponds to the number of breaks.